In [ ]:
# ============================================================
# NB_TRNSF_SilverSFTP_Person
# Capa: Silver | Origen: SFTP | Tabla: Person
# Carga TOTAL — reemplaza todo en cada ejecución
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F

# --- Config desde lh_config ---
config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen = 'SFTP'
      AND nombre_tabla = 'Person'
      AND activo = 1
    LIMIT 1
""").collect()[0]

nombre_tabla = config['nombre_tabla']
origen       = config['origen']
tipo_carga   = config['tipo_carga']
pks          = config['pks'].split(',')
params_inc   = config['parametros_incrementales']

# --- IDs técnicos (inmutables del TP) ---
# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
# lakehouse.get() funciona en pipeline y en ejecución manual.
# El pipeline setea lh_silver como defaultLakehouse de nb_silver_SFTP.
_lh_silver  = notebookutils.lakehouse.get('lh_silver')
_lh_bronze  = notebookutils.lakehouse.get('lh_bronze')
_lh_landing = notebookutils.lakehouse.get('lh_landing')
WS_ID       = _lh_silver.workspaceId
ART_SILVER  = _lh_silver.id
ART_BRONZE  = _lh_bronze.id
ART_LANDING = _lh_landing.id


abfs_silver = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_SILVER}/Tables/{origen}/{nombre_tabla}"
)

print(f"✅ Config cargada")
print(f"   nombre_tabla : {nombre_tabla}")
print(f"   origen       : {origen}")
print(f"   tipo_carga   : {tipo_carga}")
print(f"   pks          : {pks}")
print(f"   abfs_silver  : {abfs_silver}")


In [ ]:
# ============================================================
# Leer desde Bronze
# Tabla construida dinámicamente desde config
# LIMIT 1000 obligatorio durante desarrollo
# ============================================================
tabla_bronze = f"lh_bronze.{origen}.{nombre_tabla}"

df_bronze = spark.sql(f"""
    SELECT * FROM {tabla_bronze}
    LIMIT 1000
""")

print(f"✅ Filas desde Bronze: {df_bronze.count()}")
print(f"   Columnas: {df_bronze.columns}")
display(df_bronze.limit(5))


In [ ]:
# ============================================================
# Tipado y limpieza
# ============================================================
df_silver = df_bronze

# Tipado
df_silver = (df_silver
    .withColumn("BusinessEntityID", F.col("BusinessEntityID").cast("integer"))
    .withColumn("EmailPromotion",   F.col("EmailPromotion").cast("integer"))
    .withColumn("ModifiedDate",     F.col("ModifiedDate").cast("timestamp"))
)

# Limpieza — pks desde config
df_silver = df_silver.dropna(subset=pks).dropDuplicates(pks)

# Trim en strings
for c in ["PersonType","FirstName","LastName","MiddleName","Suffix","Title"]:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, F.trim(F.col(c)))

print(f"✅ Filas después limpieza: {df_silver.count()}")
display(df_silver.select("BusinessEntityID","FirstName","LastName","PersonType").limit(5))


In [ ]:
# ============================================================
# Guardar en Silver — carga total
# Borrar tabla anterior y escribir todo de nuevo
# ============================================================
try:
    notebookutils.fs.rm(abfs_silver, recurse=True)
    print(f"🗑️  Tabla anterior eliminada: {abfs_silver}")
except:
    print("ℹ️  Tabla no existía, continuando...")

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(abfs_silver))

print(f"✅ Datos guardados en Silver: {abfs_silver}")


In [ ]:
# ============================================================
# Verificar resultado en Silver
# ============================================================
df_ver = spark.read.format("delta").load(abfs_silver)

print(f"✅ Total filas en Silver ({origen}.{nombre_tabla}): {df_ver.count()}")
display(df_ver.select(
    "BusinessEntityID", "FirstName", "LastName",
    "PersonType", "EmailPromotion"
).limit(10))
